In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

# --- 1. DATA LOADING (LangChain + PyMuPDF) ---
# We use LangChain here because its PyMuPDF integration is very stable
from langchain_community.document_loaders import PyMuPDFLoader

# --- 2. LLM & EMBEDDINGS (LlamaIndex) ---
# Use LlamaIndex for models to ensure they work with the Graph & Vector Index
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Global Configuration
Settings.llm = Groq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# --- 3. CHUNKING & INDEXING (LlamaIndex) ---
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# --- 4. GRAPH RAG (LlamaIndex) ---
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex

# --- 5. AGENTIC WORKFLOW (LlamaIndex) ---
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import QueryEngineTool, ToolMetadata

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### **Vector Store Configuration**

In [3]:
# 1. Initialize Chroma Client
db = chromadb.PersistentClient(path=r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database")
chroma_collection = db.get_or_create_collection("legal_knowledge")

# 2. Initialize the Store
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# 3. Setup Storage Context
storage_context = StorageContext.from_defaults(vector_store=vector_store)

### Vector Storage Configuration: ChromaDB

This module initializes the persistent storage layer for the Legal RAG system. It ensures that embeddings generated from the Indian Constitution and case files are stored locally and are retrievable across different sessions.

#### 🛠 Configuration Details:
* **Database Type**: Persistent ChromaDB (On-disk storage).
* **Storage Path**: `C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database`
* **Collection Name**: `legal_knowledge`
* **Framework Integration**: LlamaIndex `StorageContext` wrapper.

#### 📍 Operational Role:
1. **Persistence**: Ensures that once the 100-page documents are indexed, they do not need to be re-processed or re-embedded, saving Groq API costs and time.
2. **Namespace Isolation**: The use of a specific collection (`legal_knowledge`) allows for future scaling, such as adding a separate collection for 'Private Case Files' vs. 'Public Constitutional Law.'
3. **Index Bridging**: Provides the necessary `storage_context` for the `VectorStoreIndex` to link the semantic nodes to their physical vector coordinates.

#### Implementation Note:
Ensure that the directory path exists and that the user has write permissions. If moving this project to a different machine, the `path` variable must be updated to the new local directory to maintain access to the indexed data.

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader
from llama_index.core import Document

# 1. Update File Path: Point to your specific PDF
file_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Indian_Constitution.pdf"

# 2. Load the PDF using LangChain's specialized loader
# PyMuPDF is excellent for maintaining the layout of legal documents
loader = PyMuPDFLoader(file_path)
vector_docs = loader.load()

# 3. Conversion Step: Transform LangChain format to LlamaIndex format
# This is the "Compatibility Bridge" required for your Graph and Vector Index
graph_docs = [Document(text=d.page_content, metadata=d.metadata) for d in vector_docs]

print(f"Successfully loaded and converted {len(graph_docs)} pages from the Indian Constitution.")

Successfully loaded and converted 402 pages from the Indian Constitution.


In [12]:
print(vector_docs[0])

page_content='£ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ 
[1
, 2024
]
THE CONSTITUTION OF INDIA
[As on 1st May, 2024] 
2024
GOVERNMENT OF INDIA
MINISTRY OF LAW AND JUSTICE
LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING' metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 0}


In [11]:
print(graph_docs[0])

Doc ID: bf7c07d8-883d-4637-8c18-abdae9710cd0
Text: £ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ  [1 , 2024 ] THE CONSTITUTION OF INDIA
[As on 1st May, 2024]  2024 GOVERNMENT OF INDIA MINISTRY OF LAW AND
JUSTICE LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING


In [9]:
# articles short
#Pages 3 to 31 contains the  introduction
articles_short = vector_docs[3:31]
print(articles_short[0])

page_content='THE CONSTITUTION OF INDIA
____________                                                                    
CONTENTS
__________
                                                                                          
PREAMBLE
PART I 
THE UNION AND ITS TERRITORY
ARTICLES 
1.
Name and territory of the Union.
  2. 
Admission or establishment of new States. 
[2A.         Sikkim to be associated with the Union.—Omitted.]
3.
Formation of new States and alteration of areas, boundaries or 
names of existing States.
4.
Laws made under articles 2 and 3 to provide for the amendment of 
the First and the Fourth Schedules and supplemental, incidental 
and consequential matters.
PART II
CITIZENSHIP
5.
Citizenship at the commencement of the Constitution.
6.
Rights of citizenship of certain persons who have migrated to 
India from Pakistan.
7.
Rights of citizenship of certain migrants to Pakistan.
8.
Rights of citizenship of certain persons of Indian origin residing    
outside India.
9

In [13]:
# preamble
preamble = vector_docs[31]
preamble

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 31}, page_content='THE CONSTITUTION OF INDIA \nPREAMBLE\nWE, THE PEOPLE OF INDIA, having solemnly resolved to constitute \nIndia into a \n1[SOVEREIGN SOCIALIST SECULAR DEMOCRATIC \nREPUBLIC] and to secure to all its citizens:\nJUSTICE, social, economic and political;\n \nLIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity;\nand to promote among them all\nFRATERNITY assuring the dignity of the individual and the 2[unity \nand integrity of the Nation];\nIN OUR CONS

In [14]:
articles = vector_docs[32:283]

In [15]:
print(articles[0].page_content)

2
PART I
THE UNION AND ITS TERRITORY
1. Name and territory of the Union.—(1) India, that is Bharat, 
shall be a Union of States.
1[(2) The States and the territories thereof shall be as specified in 
the First Schedule.]
(3) The territory of India shall comprise—
(a) the territories of the States; 
2[(b) the Union territories specified in the First Schedule; 
and]
(c) such other territories as may be acquired.
2. Admission or establishment of new States.—Parliament may 
by law admit into the Union, or establish, new States on such terms and 
conditions as it thinks fit.
3[2A. [Sikkim to be associated with the Union.] —Omitted by the 
Constitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).]
3. Formation of new States and alteration of areas, 
boundaries or names of existing States.—Parliament may by law—
(a) form a new State by separation of territory from any 
State or by uniting two or more States or parts of States or by 
uniting any territory to a part of any State